In [ ]:
import os

os.chdir('..')
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be set before importing torch

## Loading the Dataset:

In this section the pointcloud is loaded. The SIREN paper suggests normalizing the point coordinates as periodic activations implicitly expect a bounded input. 

In [2]:
import open3d as o3d
import numpy as np
import torch
import matplotlib.pyplot as plt
import src.model.SIREN as si
from src.model.training import train
import src.loss.SDF_loss as loss
from src.mesh_extraction.marching_cubes_test import write_obj
import src.model.MLP as simple
import src.data.dataset as data

np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

mesh = data.MeshDataset('data/pointclouds/Armadillo/Armadillo.ply')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Defining the Model

In this cell we will define the SIREN model. This particular INR uses sine activations for nonlinearity and is supposed to capture more information given the underlying data when compared to a model that uses ReLU activations. This way, a good INR accuracy can be achieved with fewer neurons.

In [3]:
size_per_layer = 256
model = si.SIRENSDF()
model_loss = loss.Loss(lambda_surface=150,  lambda_eikonal=0.5, lambda_normal=1.5, lambda_inter=0.5, lambda_sign=1.5, lambda_twd=1e-4, k=int(size_per_layer/5), model=model) # optional normal loss if normals contained in the pointcloud
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


## Model training



In [4]:
train(epochs=500, data=mesh, no_surface=10000, no_off_surface=10000, model=model, loss=model_loss, optimizer=optimizer, prune=False, scene=mesh.scene)

RuntimeError: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility

#### Model size after pruning

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Sample a few surface points and check their SDF values
test_points = mesh.vertices[:10]  # First 10 points
test_tensor = torch.from_numpy(test_points).float().to(device)
with torch.no_grad():
    sdf_values = model(test_tensor)
print("SDF values for surface points:")
print(sdf_values)
print("Mean absolute SDF:", torch.abs(sdf_values).mean().item())

SDF values for surface points:
tensor([[ 0.0127],
        [-0.0181],
        [ 0.0136],
        [ 0.0155],
        [-0.0178],
        [-0.0178],
        [-0.0180],
        [ 0.0125],
        [-0.0140],
        [-0.0147]], device='cuda:0')
Mean absolute SDF: 0.015470693819224834


In [ ]:
import src.mesh_extraction.marching_cubes_gpu as marching_cubes
marching_cubes.write_obj("armadillo_64_reproducibility_test_1.obj", model=model, resolution=64, level=0.0)


In [ ]:
# import src.mesh_extraction.marching_cubes_test as marching_cubes_test
# marching_cubes_test.write_obj("armadillo_mesh_ground_truth_128.obj", scene=mesh.scene, resolution=128, level=0.0)


In [ ]:
# Make batched point
test_point = torch.from_numpy(np.array([[-1, -1, -1]])).float().to(device)

# Compute SIREN model prediction
sdf_pred = model(test_point)
print("SIREN prediction:", sdf_pred)

# Compute Open3D signed distance
distance = mesh.scene.compute_signed_distance(
    o3d.core.Tensor(test_point.cpu().numpy(), dtype=o3d.core.Dtype.Float32)
)
print("Open3D SDF:", distance.numpy())

SIREN prediction: tensor([[0.0593]], device='cuda:0', grad_fn=<AddmmBackward0>)
Open3D SDF: [0.96991843]
